In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from itertools import chain


/scratch2/mrenaudin/GTH/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("text", data_files={
    "train": "/scratch2/mrenaudin/GTH/data/english_data/train.txt",
    "validation": "/scratch2/mrenaudin/GTH/data/english_data/valid.txt",
}, cache_dir=".cache")

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2", cache_dir=".cache")
tokenizer.pad_token = tokenizer.eos_token


In [4]:
def tokenize(batch):
    return tokenizer(batch["text"])

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])


In [5]:
block_size =512
def group_texts(examples):
        # Concatenate all texts.
        concatenated_examples = {k: list(chain(*examples[k])) for k in examples}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # We drop the small remainder, and if the total_length < block_size  we exclude this batch and return an empty dict.
        # We could add padding if the model supported it instead of this drop, you can customize this part to your needs.
        total_length = (total_length // block_size) * block_size
        # Split by chunks of max_len.
        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

In [6]:
lm_datasets = tokenized.map(
                group_texts,
                batched=True,
                num_proc=4,
                desc=f"Grouping texts in chunks of {block_size}",
            )

In [7]:
lm_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 197345
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 24690
    })
})

In [8]:
config = AutoConfig.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_config(config)

In [9]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [11]:
training_args = TrainingArguments(
    output_dir="gpt2-out",
    per_device_train_batch_size=4,
    # evaluation_strategy="steps",
    eval_steps=1,
    save_steps=1,
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

[RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.16s/it]


KeyboardInterrupt: 

In [ ]:
%pip install accelerate>=1.1.0

Note: you may need to restart the kernel to use updated packages.
